# 02 — Deployment profile

**docs/04 §1.** What each project-sensor deployment recorded over the window the
logger was in the water, and how it compared to its nearest public reference.

**This describes; it tests nothing.** There is no hypothesis here and no
inference. Every number below is a summary of one deployment or one comparison
already computed by the pipeline — this notebook arranges them, states what they
do and do not support, and stops. The analysis ladder starts at docs/04 §4.1 and
this is not a rung of it.

**Why a deployment and not a quarter.** `quarterly_env` measures every series
against the Kelp Watch calendar, which is right for the kelp comparison and wrong
for a logger that was down for three weeks: it reads such a deployment at 0.228
coverage, marks it unusable, and flags a warm spell running to the start of the
record as a floor that may have been longer. It may not have been — nobody was
measuring. `deployment.parquet` (docs/03) is the same docs/04 §2 arithmetic over
the window the registry says the instrument was actually down.

**Why a day as well as a deployment.** A deployment row says the shallower logger
accumulated 78.93 °C·days above 18 °C. It cannot say whether that arrived
steadily or in one week, and until `deployment_daily.parquet` nothing in
`features/` could: the finest grain the zone exposed was the deployment window,
while the instrument samples every 600 s. Whether the zone should reach further
down still, and on what rule, is
[#158](https://github.com/cweber12/kelp-compare/issues/158).

**Reproducibility.** Runs top to bottom from `features/deployment.parquet`,
`features/deployment_daily.parquet` and `features/validation.parquet`, and
nothing else — no side reads of `observations/` or `raw/`. Nothing here is
stochastic, so there is no seed to set. All three tables are stamped with their
SHA-256 below, and every figure carries the digest of the table it was drawn
from; quote them in any caption or write-up.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt

from kelpcompare.features.deployment import DEPLOYMENT_DAILY_KEY, DEPLOYMENT_KEY
from kelpcompare.features.validation import VALIDATION_KEY
from kelpcompare.figures import Band, plot_bands

pd.set_option("display.width", 200)


def repo_root() -> Path:
    """The checkout root, so the notebook runs from anywhere."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "features" / "deployment.parquet").exists():
            return candidate
    raise FileNotFoundError(
        "no data/features/deployment.parquet above the working directory -- "
        "run `kelpcompare deployments` first"
    )


ROOT = repo_root() / "data" / "features"
FIGURES = repo_root() / "notebooks" / "figures"
TABLES = {
    "deployment": ROOT / "deployment.parquet",
    "deployment_daily": ROOT / "deployment_daily.parquet",
    "validation": ROOT / "validation.parquet",
}
DIGESTS = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in TABLES.items()
    if path.exists()
}


def caption(*names: str) -> str:
    """What a figure was drawn from, in the figure (notebooks/README).

    A figure that leaves the repo on its own still has to say which table
    produced it, so the digest travels in the caption rather than beside it.
    """
    return "   ".join(f"{name}.parquet sha256:{DIGESTS[name][:16]}" for name in names)


deployment = pd.read_parquet(TABLES["deployment"])
daily = pd.read_parquet(TABLES["deployment_daily"])

# Which columns identify one deployment is a fact about the schema (docs/03), so
# it is read from the package that wrote the table rather than restated here. A
# restated key is how an analysis silently pools two depths of one logger into
# one row the first time a site carries a second instrument.
absent = [column for column in DEPLOYMENT_KEY if column not in deployment.columns]
absent += [column for column in DEPLOYMENT_DAILY_KEY if column not in daily.columns]
if absent:
    raise ValueError(
        f"the installed kelpcompare does not describe these tables -- {absent} absent. "
        "Rebuild them with `kelpcompare deployments`, or check out the package these "
        "tables were written by."
    )

# The daily table is the deployment key plus the day, so every daily row belongs
# to exactly one deployment row. Asserted rather than assumed: if that stopped
# being true the audit in section 4 would be comparing two different groupings
# and would still print numbers.
assert list(DEPLOYMENT_DAILY_KEY) == [*DEPLOYMENT_KEY, "day"], DEPLOYMENT_DAILY_KEY

# The join to validation is on the deployment, which is VALIDATION_KEY without its
# two reference columns -- derived rather than typed out, so that a schema change
# on either side surfaces here instead of silently dropping rows.
JOIN_KEY = [name for name in VALIDATION_KEY if not name.startswith("reference_")]
assert JOIN_KEY == list(DEPLOYMENT_KEY), (JOIN_KEY, DEPLOYMENT_KEY)

for name, digest in DIGESTS.items():
    print(f"{name + '.parquet':>28}  {digest}")
if "validation" not in DIGESTS:
    print("\nvalidation.parquet absent -- section 7 will report nothing.")
print(f"\n{len(deployment)} deployment rows over {len(daily)} observed days")

          deployment.parquet  9b16925340fd7b0a155d70f2b7d4cede7a812e36abe9ac00a6b952e9eaef04a4
    deployment_daily.parquet  eb1d836da77dc04c93000146489e2ad6f38d3e533a85064040639361462bec34
          validation.parquet  3d7aea1aaf904fcf0b97b0124f743ccfd443e2ad3d9333835f00febbaa357461

2 deployment rows over 66 observed days


## 1. The gate

`deployment.parquet` keeps every registered deployment that produced rows,
including those that did not clear the coverage floor — docs/03 makes `usable` a
flag rather than a filter, so what filtering costs is visible rather than already
spent.

Spending it here, once, and reporting what it cost.

**What `usable` means here is not what it means quarterly.** Measured against a
deployment's own window, coverage below the floor is not a calendar artifact: it
is the instrument stopping early, flooding, or failing QC. A `false` in this
table is a statement about the logger.

In [2]:
attrition = pd.Series(
    {
        "deployments with rows": len(deployment),
        "...complete (recovered, not still down)": int(deployment["deployment_complete"].sum()),
        "...usable (cleared the coverage floor)": int(deployment["usable"].sum()),
    },
    name="rows",
)
print(attrition.to_string())

gated = deployment[deployment["usable"]]
lost = deployment[~deployment["usable"]]
if len(lost):
    print()
    print("Excluded, with the coverage that excluded them:")
    print(
        lost[["site_id", "deployment_number", "pct_coverage", "n_obs", "expected_obs"]].to_string(
            index=False
        )
    )
else:
    print("\nNothing excluded: every deployment cleared the floor.")

deployments with rows                      2
...complete (recovered, not still down)    2
...usable (cleared the coverage floor)     2

Nothing excluded: every deployment cleared the floor.


## 2. What each deployment recorded

The window, what arrived in it, and the distribution of the water while it was
there.

`pct_coverage` is `n_obs / expected_obs` where `expected_obs` is the window's
duration over the series' median native cadence, **plus one** — a closed window
sampled every *c* seconds across *D* seconds holds `D/c + 1` readings, not `D/c`
(docs/03). So a healthy deployment reads 1.000 rather than fractionally over.

In [3]:
window = gated.assign(
    days=(gated["window_end"] - gated["window_start"]).dt.total_seconds() / 86400,
)
print(
    window[
        [
            "site_id",
            "deployment_number",
            "depth_m",
            "window_start",
            "window_end",
            "days",
            "n_obs",
            "expected_obs",
            "pct_coverage",
            "cadence_s",
        ]
    ]
    .round({"days": 1, "pct_coverage": 4})
    .to_string(index=False)
)
print()
print(
    gated[["site_id", "depth_m", "mean", "min", "p05", "p95", "max", "variance"]]
    .round(2)
    .to_string(index=False)
)

      site_id  deployment_number  depth_m        window_start          window_end  days  n_obs  expected_obs  pct_coverage  cadence_s
PROJ:TIDBIT-1                  3     8.23 2026-07-11 15:00:00 2026-08-01 14:30:00  21.0   3022        3022.0        1.0000      600.0
PROJ:TIDBIT-2                  3    16.76 2026-07-11 14:40:00 2026-08-23 15:30:00  43.0   6194        6198.0        0.9994      600.0

      site_id  depth_m  mean   min   p05   p95   max  variance
PROJ:TIDBIT-1     8.23 21.58 17.76 19.21 23.62 24.08      1.73
PROJ:TIDBIT-2    16.76 16.50 13.12 14.27 19.45 23.02      2.72


## 3. The record, drawn

The table above reduces each deployment to one row. This is what those rows are
summaries of: the daily mean inside the day's minimum and maximum, from
`deployment_daily.parquet`.

**The line is cut wherever a day is missing rather than drawn through it.** The
interval between two rows of that table is a day nobody measured, and bridging it
would assert a value for that day — the same reading docs/04 §2 gives a gap when
it refuses to join a spell across one.

**A day clipped by the deployment boundary is a whole day, not a short one.** The
shallower logger went in at 15:00 UTC and recorded every one of the nine hours
available to it; judged against a full 24 that first day would read 0.375 covered
and look like a fault, which is precisely the mistake the quarterly calendar
makes on a whole deployment. The daily table clips each day to the window and
marks it `partial_day`, so the boundary is visible without being counted as loss.

**These two lines are not a profile.** The instruments sit 456 m apart, and the
deeper one stayed down three weeks longer. They share a time axis because they
share a summer, not because they share a water column.

In [4]:
# Only the deployments section 1 kept. The gate is applied once, here, and every
# figure below is drawn from `kept` -- an unusable deployment silently entering
# one figure and not another is how two figures in one document come to disagree.
kept = daily.merge(gated[list(DEPLOYMENT_KEY)], on=list(DEPLOYMENT_KEY), how="inner")


def by_depth(frame: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
    """One (label, days) pair per deployment, shallowest first.

    The order is the figure's colour ramp (`plot_bands` shades by position), so
    sorting by depth is what makes pale-to-dark read as shallow-to-deep.
    """
    groups = sorted(frame.groupby(["site_id", "depth_m"]), key=lambda item: item[0][1])
    return [(f"{site} at {depth:.2f} m", part.sort_values("day")) for (site, depth), part in groups]


profile = [
    Band(label=label, x=part["day"], centre=part["mean"], low=part["min"], high=part["max"])
    for label, part in by_depth(kept)
]

figure = plot_bands(
    profile,
    title="What each deployment recorded, day by day",
    subtitle=(
        "Daily mean inside the daily minimum and maximum; a missing day breaks the line. "
        "Different positions over different windows -- not one profile."
    ),
    ylabel="sea water temperature (degC)",
    caption=caption("deployment_daily"),
    gap=pd.Timedelta(days=1),
)
path = FIGURES / "DEPLOYMENT-PROFILE.png"
figure.savefig(path, facecolor=figure.get_facecolor())
plt.close(figure)
print(f"wrote {path.relative_to(repo_root())}")

partial = kept[kept["partial_day"]]
print(f"\n{len(partial)} days clipped by a deployment boundary, and their coverage:")
print(
    partial[["site_id", "day", "n_obs", "expected_obs", "pct_coverage"]]
    .round({"pct_coverage": 4})
    .to_string(index=False)
)

wrote notebooks\figures\DEPLOYMENT-PROFILE.png

4 days clipped by a deployment boundary, and their coverage:
      site_id        day  n_obs  expected_obs  pct_coverage
PROJ:TIDBIT-1 2026-07-11     54          54.0        1.0000
PROJ:TIDBIT-1 2026-08-01     88          88.0        1.0000
PROJ:TIDBIT-2 2026-07-11     53          56.0        0.9464
PROJ:TIDBIT-2 2026-08-23     94          94.0        1.0000


## 4. Thermal exposure, and what depth does to it

The docs/04 §2 features are day-based: a day that reached the threshold is a day
of exposure whether it did so for one hour or six. Every day with at least one
observation counts, and `n_days_observed` records how many that was — **so every
count below is a floor rather than a census**, and it runs low under partial
coverage because a day observed only overnight cannot show its daytime maximum.

`max_spell_above_20c_gap_interrupted` says whether the longest run ended at an
unobserved day, meaning the true spell may have been longer. Against a deployment
window it should normally be `False`: the days before the logger went in are not
a gap in the record.

In [5]:
EXPOSURE = [
    column
    for column in ("days_above_20c", "days_above_23c", "days_below_14c", "degree_days_above_18c")
    if column in gated.columns
]
SPELL = [column for column in gated.columns if column.startswith("max_spell_above_")]

exposure = gated[["site_id", "depth_m", "n_days_observed", *EXPOSURE, *SPELL]].copy()
for column in EXPOSURE:
    if column.startswith("days_"):
        exposure[column + "_frac"] = (gated[column] / gated["n_days_observed"]).round(2)
print(exposure.round(2).to_string(index=False))

print()
for row in gated.to_dict("records"):
    share = (
        row["days_above_20c"] / row["n_days_observed"] if row["n_days_observed"] else float("nan")
    )
    print(
        f"{row['site_id']:>16} at {row['depth_m']:>6.2f} m: "
        f"{int(row['days_above_20c'])} of {int(row['n_days_observed'])} observed days "
        f"reached 20 degC ({share:.0%}), {int(row['days_below_14c'])} fell below 14 degC."
    )

      site_id  depth_m  n_days_observed  days_above_20c  days_above_23c  days_below_14c  degree_days_above_18c  max_spell_above_20c_days  max_spell_above_20c_gap_interrupted  days_above_20c_frac  days_above_23c_frac  days_below_14c_frac
PROJ:TIDBIT-1     8.23               22            22.0            10.0             0.0                  78.93                      22.0                                False                 1.00                 0.45                 0.00
PROJ:TIDBIT-2    16.76               44            12.0             1.0            14.0                   4.00                       5.0                                False                 0.27                 0.02                 0.32

   PROJ:TIDBIT-1 at   8.23 m: 22 of 22 observed days reached 20 degC (100%), 0 fell below 14 degC.
   PROJ:TIDBIT-2 at  16.76 m: 12 of 44 observed days reached 20 degC (27%), 14 fell below 14 degC.


### The counts, re-derived from the days they are counts of

`days_above_20c` is the number of observed days whose daily maximum exceeded
20 °C. `deployment_daily.parquet` holds those daily maxima, so the count can be
reproduced rather than trusted — and so can the coverage arithmetic, because the
clipped days tile the deployment window exactly (docs/03).

**This cell raises if the two tables disagree**, which is the whole reason it is
here. They are written by one command over one walk of the registry, so a
disagreement would mean they had drifted apart — and the way that gets
discovered otherwise is somebody quoting one of them.

In [6]:
# The thresholds come from the column names the pipeline derived from
# features.json (docs/03), so a retune renames the column and this audit follows
# it rather than checking a number nothing produces any more.
audit = (
    kept.groupby(list(DEPLOYMENT_KEY), dropna=False)
    .agg(
        n_days_observed=("day", "size"),
        n_obs=("n_obs", "sum"),
        expected_obs=("expected_obs", "sum"),
        days_above_20c=("max", lambda days: float((days > 20.0).sum())),
        days_above_23c=("max", lambda days: float((days > 23.0).sum())),
        days_below_14c=("min", lambda days: float((days < 14.0).sum())),
        degree_days_above_18c=("mean", lambda days: float((days - 18.0).clip(lower=0.0).sum())),
    )
    .reset_index()
)

checked = gated.merge(audit, on=list(DEPLOYMENT_KEY), suffixes=("", "_daily"), validate="1:1")
disagreed = []
for column in [name for name in audit.columns if name not in DEPLOYMENT_KEY]:
    scalar, rebuilt = checked[column], checked[f"{column}_daily"]
    close = (scalar - rebuilt).abs() < 1e-9
    print(f"{column:>24}  {'agrees' if close.all() else 'DISAGREES'}")
    for row, ok in zip(checked.to_dict("records"), close, strict=True):
        if not ok:
            disagreed.append(
                f"{row['site_id']}: {column} {row[column]} vs {row[column + '_daily']}"
            )

if disagreed:
    raise AssertionError(
        "deployment.parquet and deployment_daily.parquet disagree about the same "
        "record:\n  " + "\n  ".join(disagreed)
    )
print(f"\n{len(checked)} deployments reproduced from {len(kept)} daily rows.")

         n_days_observed  agrees
                   n_obs  agrees
            expected_obs  agrees
          days_above_20c  agrees
          days_above_23c  agrees
          days_below_14c  agrees
   degree_days_above_18c  agrees

2 deployments reproduced from 66 daily rows.


### The depth contrast is the one thing here worth calling a finding

Two loggers, one summer, and a difference large enough that it does not need a
test to be visible. The shallower sits above the seasonal thermocline and the
deeper below it, and the ecological features say so more plainly than the means
do — degree-days above 18 °C differ by more than an order of magnitude.

**The deeper logger is nonetheless the more variable one**, on variance and on
daily range alike, which is the opposite of what a thermally buffered position
would suggest and is visible in the figures in section 5. Reading that as
evidence of any particular mechanism would be over-reading it: these are two
different windows, and the deeper record is three weeks longer.

**What it is not.** These are two *different windows*, so this compares
deployments, not simultaneous water. The paired, same-instant comparison that
would separate stratification from the extra weeks is a cross-sensor statistic
neither of these tables can express, and building one needs a rule for which
sensors may validly be compared that the registry does not yet have
(https://github.com/cweber12/kelp-compare/issues/71). It is also two *positions*
456 m apart, not one profile.

In [7]:
CONTRASTED = (
    ("mean", "mean"),
    ("p95", "p95"),
    ("min", "min"),
    ("variance", "variance"),
    ("degree-days >18 degC", "degree_days_above_18c"),
)

# The shallowest against the deepest, whatever the registry holds. This used to
# assume exactly two deployments, which would have started reporting the wrong
# pair the first time a third logger landed rather than saying it could not.
if len(gated) >= 2:
    ranked = gated.sort_values("depth_m").to_dict("records")
    shallow, deep = ranked[0], ranked[-1]
    print(
        f"{shallow['site_id']} at {shallow['depth_m']} m vs "
        f"{deep['site_id']} at {deep['depth_m']} m"
    )
    if len(ranked) > 2:
        between = ", ".join(f"{row['site_id']} at {row['depth_m']} m" for row in ranked[1:-1])
        print(f"the extremes of {len(ranked)} usable deployments; also held: {between}")
    print(
        f"depth gap {deep['depth_m'] - shallow['depth_m']:.2f} m, and NOT one profile -- "
        "different positions, different windows\n"
    )
    for label, key in CONTRASTED:
        if key in gated.columns:
            print(f"  {label:>22}: {shallow[key]:>8.2f}  vs {deep[key]:>8.2f}")
else:
    print(f"{len(gated)} usable deployment; a depth contrast needs at least two.")

PROJ:TIDBIT-1 at 8.23 m vs PROJ:TIDBIT-2 at 16.76 m
depth gap 8.53 m, and NOT one profile -- different positions, different windows

                    mean:    21.58  vs    16.50
                     p95:    23.62  vs    19.45
                     min:    17.76  vs    13.12
                variance:     1.73  vs     2.72
    degree-days >18 degC:    78.93  vs     4.00


## 5. How the heat arrived, and how far the water moved each day

Two readings a scalar row cannot give.

**Accumulation.** Degree-days above 18 °C is a sum over observed days of the
positive excess of that day's mean (docs/04 §2). 78.93 against 4.00 is a ratio;
the curve is a shape, and it says whether the total arrived steadily or in an
event. The final value of each curve is the scalar the section above audited.

**Daily range.** The daily maximum minus the daily minimum. It is the cheapest
observable bearing on how far the water at a fixed depth moved within a day, and
it is worth reading beside
[#73](https://github.com/cweber12/kelp-compare/issues/73) — the 5.0 m neighbour
depth tolerance is a stratification threshold set on two pairs from one summer,
and a winter deployment is what would retune it.

**It is not a thermocline measurement and nothing here corrects for anything.** A
large daily range at a fixed depth is consistent with vertical movement of the
water column past the sensor, with surface heating, and with several other
things. Distinguishing them needs a covariate this project does not yet hold —
tide, most obviously — and no such claim is made below.

In [8]:
DEGREE_DAY_BASE = 18.0

accumulation, spread = [], []
for label, part in by_depth(kept):
    excess = (part["mean"] - DEGREE_DAY_BASE).clip(lower=0.0)
    accumulation.append(Band(label=label, x=part["day"], centre=excess.cumsum()))
    spread.append(Band(label=label, x=part["day"], centre=part["max"] - part["min"]))

for bands, name, title, ylabel, subtitle in (
    (
        accumulation,
        "DEPLOYMENT-ACCUMULATION",
        f"How the heat above {DEGREE_DAY_BASE:.0f} degC accumulated",
        "cumulative degree-days (degC-day)",
        "Each curve ends at the deployment row's degree_days_above_18c.",
    ),
    (
        spread,
        "DEPLOYMENT-DAILY-RANGE",
        "How far the water moved within each day",
        "daily maximum minus minimum (degC)",
        "Descriptive. Not a thermocline measurement, and corrected for nothing.",
    ),
):
    figure = plot_bands(
        bands,
        title=title,
        subtitle=subtitle,
        ylabel=ylabel,
        caption=caption("deployment_daily"),
        gap=pd.Timedelta(days=1),
    )
    written = FIGURES / f"{name}.png"
    figure.savefig(written, facecolor=figure.get_facecolor())
    plt.close(figure)
    print(f"wrote {written.relative_to(repo_root())}")

print()
for label, part in by_depth(kept):
    reach = (part["mean"] - DEGREE_DAY_BASE).clip(lower=0.0).sum()
    daily_range = part["max"] - part["min"]
    print(
        f"{label:>26}: {reach:6.2f} degC-day accumulated over "
        f"{len(part):>2} days, daily range {daily_range.mean():.2f} degC mean, "
        f"{daily_range.min():.2f}-{daily_range.max():.2f} observed."
    )

wrote notebooks\figures\DEPLOYMENT-ACCUMULATION.png
wrote notebooks\figures\DEPLOYMENT-DAILY-RANGE.png

   PROJ:TIDBIT-1 at 8.23 m:  78.93 degC-day accumulated over 22 days, daily range 3.15 degC mean, 0.76-5.50 observed.
  PROJ:TIDBIT-2 at 16.76 m:   4.00 degC-day accumulated over 44 days, daily range 4.26 degC mean, 1.72-6.82 observed.


## 6. How each deployment compared to its neighbour

`validation.parquet` (docs/04 §1), joined on the deployment. **Bias and RMSE are
null across a depth gap wider than the configured tolerance, and that is a
refusal rather than missing data** — below the thermocline the offset between two
depths *is* most of the signal, so a bias computed across one measures
stratification and prints it as instrument error. `n_pairs` is populated either
way, which is how the two are told apart.

In [9]:
if "validation" in DIGESTS:
    validation = pd.read_parquet(TABLES["validation"])
    joined = gated.merge(validation, on=JOIN_KEY, how="left", suffixes=("", "_val"))
    print(
        joined[
            [
                "site_id",
                "depth_m",
                "reference_site_id",
                "reference_depth_m",
                "depth_gap_m",
                "depth_comparable",
                "n_pairs",
                "correlation",
                "bias",
                "rmse",
                "collapsed_refs",
            ]
        ]
        .round(3)
        .to_string(index=False)
    )
    print()
    for row in joined.to_dict("records"):
        if pd.isna(row.get("reference_site_id")):
            print(f"{row['site_id']}: no validation row -- no reference with overlapping data.")
        elif row["depth_comparable"]:
            print(
                f"{row['site_id']}: r = {row['correlation']:.3f} against "
                f"{row['reference_site_id']}, bias {row['bias']:+.2f} degC across a "
                f"{row['depth_gap_m']:.2f} m gap."
            )
        else:
            print(
                f"{row['site_id']}: r = {row['correlation']:.3f} against "
                f"{row['reference_site_id']}; bias and RMSE refused across "
                f"{row['depth_gap_m']:.2f} m."
            )
else:
    print("validation.parquet absent -- run `kelpcompare validate`.")

      site_id  depth_m reference_site_id  reference_depth_m  depth_gap_m  depth_comparable  n_pairs  correlation   bias  rmse collapsed_refs
PROJ:TIDBIT-1     8.23        NDBC:LJAC1                3.4         4.83              True     2636        0.643 -1.037 1.543  COOPS:9410230
PROJ:TIDBIT-2    16.76        NDBC:LJAC1                3.4        13.36             False     5786        0.641    NaN   NaN  COOPS:9410230

PROJ:TIDBIT-1: r = 0.643 against NDBC:LJAC1, bias -1.04 degC across a 4.83 m gap.
PROJ:TIDBIT-2: r = 0.641 against NDBC:LJAC1; bias and RMSE refused across 13.36 m.


## 7. What this does not say

Carried from docs/04 §1 and §6, because a descriptive table is where
over-reading is easiest:

- **No anomalies, and none are coming from this record.** A climatology needs ten
  usable years inside 2007–2019 (docs/04 §3) and ADR-007 makes that minimum
  non-overridable, so nothing here is expressed relative to normal. Every number
  above is an absolute summary of one summer. Neither `deployment.parquet` nor
  `deployment_daily.parquet` carries an `_anom` column at all so that this cannot
  be forgotten.
- **A high correlation with a neighbour is not evidence of local signal.** It is
  evidence the instrument works. The more interesting claim — that these sensors
  capture something the public network misses — is docs/04 §4.5, and it is
  deferred on sample size, not answerable here
  (https://github.com/cweber12/kelp-compare/issues/120).
- **Correlation against a reference at another depth degrades with the gap, and
  must be read with it.** Both loggers here correlate with `NDBC:LJAC1` at almost
  the same value despite gaps of 4.83 m and 13.36 m, which docs/04 §1 predicts
  should not happen and is an open question
  (https://github.com/cweber12/kelp-compare/issues/74), not a result. **Nothing
  added here settles it.** The proposed settlement is to recompute both pairs at
  more than one averaging timescale, which needs the paired series that
  `validation.parquet` bins and then discards — a change to that table, not a
  reading of this one.
- **Every day-based count is a floor.** Partial coverage biases them low, in the
  direction docs/04 §2 documents. The daily table makes the floor visible rather
  than removing it: a day is still a day whether it held 144 observations or 2.
- **A daily range is not a stratification measurement.** It is consistent with
  vertical movement of the column past a fixed sensor, with surface heating, and
  with other things. Separating them needs a covariate this project does not hold
  — tide is the obvious one, and there is no CO-OPS fetcher.
- **One summer, two instruments.** The depth tolerance that decides which
  comparisons report a bias is itself set on this evidence
  (https://github.com/cweber12/kelp-compare/issues/73), so it should be retuned
  against a record spanning a winter, when the column is mixed and a 13 m gap may
  be no gap at all. Nothing here can be read as confirming it.